# Baseline transcription

Transcribes 1,200 FLEURS clips and 1,200 UrduSpeech news + podcast clips with three recognisers and scores them against the references.

| Recogniser | Model | Environment |
|---|---|---|
| Whisper-large-v3 | `openai/whisper-large-v3` | Python 3.10 |
| Omnilingual LLM 1B v2 | `omniASR_LLM_1B_v2` | Python 3.10 (Omnilingual has no build for newer Python versions) |
| Indic Transcribe Core | `bodhan-ai/indic-transcribe-core` | Default Python |

Outputs, in the repository layout:

| File | Contents |
|---|---|
| `data/samples_FLEURS.csv`, `data/samples_UrduSpeech_news_podcast.csv` | One row per clip: `clip_id`, dataset, subset, category, duration, reference, whether it is transcribed |
| `data/transcripts/<recogniser>_transcripts.csv` | One row per clip: transcript, seconds taken, status, decoding settings, word confidence. Saved every 50 clips; also the checkpoint. |
| `data/transcripts/all_recognisers_by_clip.csv` | One row per clip with the reference and all three transcripts, raw and normalised, with per-clip WER and CER |
| `experiments/01_baseline_transcription/results/scores/recogniser_scores.csv` | WER, CER, substitution / deletion / insertion %, and time for each dataset subset |
| `experiments/01_baseline_transcription/results/scores/paired_bootstrap.csv` | 95% interval of the WER difference between every pair of recognisers |
| `experiments/01_baseline_transcription/results/run_info.json` | GPU, package versions, models, decoding settings, clip counts, elapsed time |

## 1. Setup

In [ ]:
import os, sys, json, glob, time, shutil, subprocess
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

def sh(cmd):
    subprocess.run(cmd, shell=True, check=True)

sh("pip install -q jiwer soundfile librosa sentencepiece")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader")

### Settings, normaliser and checkpoint helpers

In [ ]:
%%writefile baseline_common.py
import os, re, glob, json, time, shutil, string, unicodedata
import pandas as pd

OUT = "/kaggle/working/output"
DATA = f"{OUT}/data"
RESULTS = f"{OUT}/experiments/01_baseline_transcription/results"
AUDIO_DIR = "/tmp/audio_16k"   # rebuilt each session; not kept as output
SAMPLE_FILES = {"FLEURS": f"{DATA}/samples_FLEURS.csv",
                "UrduSpeech": f"{DATA}/samples_UrduSpeech_news_podcast.csv"}
SAMPLE_RATE = 16000
MAX_CLIP_SECONDS = 40          # Omnilingual's limit, applied to all recognisers so they score the same clips
CHECKPOINT_EVERY = 50

RECOGNISERS = {
    "whisper_large_v3": "Whisper-large-v3",
    "omnilingual_llm_1b_v2": "Omnilingual LLM 1B v2",
    "indic_transcribe_core": "Indic Transcribe Core",
}
TRANSCRIPT_COLUMNS = ["clip_id", "transcript", "time_s", "status", "decoding", "word_confidence"]

# clean_urdu v1
NORMALISER_VERSION = "clean_urdu v1"
_DIAC_RE = re.compile("[ؐ-ؚـً-ٰۖ-ۜ۟-۪ۨ-ۭ]")
_PUNCT = set(string.punctuation) | set("۔،؟؛٪।॥“”"
                                       "‘’—–…«»‖‐‒·")
_IZAFAT = {"اے", "ای", "ئے", "ء", "ے", "او"}
_UMAP = {"ي": "ی", "ك": "ک", "ة": "ہ", "ۃ": "ہ",
         "ى": "ی", "إ": "ا", "أ": "ا", "آ": "ا",
         "ٲ": "ا", "ٳ": "ا", "ؤ": "و", "ئ": "ی"}

def clean_urdu(text):
    t = unicodedata.normalize("NFC", str(text))
    t = _DIAC_RE.sub("", t)
    t = "".join(_UMAP.get(ch, ch) for ch in t)
    t = "".join(" " if ch in _PUNCT else ch for ch in t)
    return " ".join(w for w in t.split() if w not in _IZAFAT).strip()


def audio_path(clip_id):
    return f"{AUDIO_DIR}/{clip_id.replace('/', '__')}.wav"


def load_samples(included_only=False):
    clips = pd.concat([pd.read_csv(p, keep_default_na=False) for p in SAMPLE_FILES.values()], ignore_index=True)
    if included_only:
        clips = clips[clips.included.astype(str) == "True"].reset_index(drop=True)
    return clips


def transcript_path(stem):
    return f"{DATA}/transcripts/{stem}_transcripts.csv"


def load_transcripts(stem):
    # Transcripts of a previous run attached as a Kaggle input are copied in first, so the run continues from them.
    path = transcript_path(stem)
    if not os.path.exists(path):
        earlier = glob.glob(f"/kaggle/input/**/data/transcripts/{stem}_transcripts.csv", recursive=True)
        if earlier:
            os.makedirs(os.path.dirname(path), exist_ok=True)
            shutil.copy(earlier[0], path)
            print(f"Continuing {RECOGNISERS[stem]} from {earlier[0]}")
    if os.path.exists(path):
        return pd.read_csv(path, keep_default_na=False)
    return pd.DataFrame(columns=TRANSCRIPT_COLUMNS)


def append_transcripts(stem, rows):
    path = transcript_path(stem)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame(rows, columns=TRANSCRIPT_COLUMNS).to_csv(
        path, mode="a", header=not os.path.exists(path), index=False, encoding="utf-8")


def run_recogniser(stem, transcribe_one):
    # transcribe_one(path, duration_s) returns {text, decoding, word_confidence, status};
    # an exception is saved as status 'error: ...' with an empty transcript.
    clips = load_samples(included_only=True)
    done = set(load_transcripts(stem).clip_id)
    todo = clips[~clips.clip_id.isin(done)]
    name = RECOGNISERS[stem]
    print(f"{name}: {len(done)} clips already done, {len(todo)} to go")
    rows, t_start = [], time.perf_counter()
    for k, clip in enumerate(todo.itertuples(), 1):
        t0 = time.perf_counter()
        try:
            r = transcribe_one(audio_path(clip.clip_id), float(clip.duration_s))
            status = r["status"] if r["text"].strip() else "empty_output"
            row = [clip.clip_id, r["text"], 0.0, status, r["decoding"],
                   json.dumps(r["word_confidence"], ensure_ascii=False) if r["word_confidence"] else ""]
        except Exception as e:
            row = [clip.clip_id, "", 0.0, f"error: {type(e).__name__}: {e}"[:300], "", ""]
        row[2] = round(time.perf_counter() - t0, 3)
        rows.append(row)
        if len(rows) == CHECKPOINT_EVERY or k == len(todo):
            append_transcripts(stem, rows)
            rows = []
            rate = (time.perf_counter() - t_start) / k
            print(f"  {name}: {len(done) + k}/{len(clips)} clips saved | {rate:.2f} s per clip", flush=True)
    print(f"{name}: finished")

### Python 3.10 environment for Omnilingual and Whisper
Built once per session.

In [ ]:
PY = "/kaggle/working/py310/bin/python"
if not os.path.exists(PY):
    sh("pip install -q uv && uv python install 3.10 && uv venv --seed --python 3.10 /kaggle/working/py310")
    sh(f"{PY} -m pip install -q --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchaudio==2.8.0")
    sh(f"{PY} -m pip install -q --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.8.0/cu128 fairseq2n==0.6.0 fairseq2==0.6.0")
    sh(f"{PY} -m pip install -q --ignore-requires-python --no-deps omnilingual-asr==0.2.0")
    sh(f'{PY} -m pip install -q polars sentencepiece editdistance retrying "pandas==2.2.3" "numpy>=1.26,<2.3" '
       f'"pyarrow==20.0.0" "fsspec<=2024.9.0" "datasets==3.2.0" "transformers>=4.46,<4.49" jiwer librosa soundfile accelerate')
    sh(f'{PY} -m pip install -q "huggingface_hub>=0.32,<0.33"')  # fairseq2 needs huggingface_hub 0.32.x; install last

sh(f'{PY} -c "from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline; '
   f"import torch, transformers; print('Python 3.10 env OK | torch', torch.__version__, '| transformers', transformers.__version__)\"")

## 2. Samples and audio

- **FLEURS (`ur_pk`):** test, validation and train joined in that order, shuffled with seed 42, first 1,200 kept.
  `datasets.shuffle(seed=42)` uses `np.random.default_rng(42).permutation`, so the same permutation is applied here directly.
- **UrduSpeech:** news and podcast JSONL files under `benchmark/US-benchmark-Std/` then `corpus/US-Std/`, read in repository order, first 1,200 clips with text and audio.

Each clip gets a `clip_id` made from its audio file path, so it can be traced back to the source dataset.
Audio is saved as 16 kHz mono 32-bit float WAV, so all recognisers read identical audio.
Clips longer than 40 s are listed with `included = False` and are not transcribed by any recogniser.
The clips are saved to `data/samples_FLEURS.csv` and `data/samples_UrduSpeech_news_podcast.csv`.

In [ ]:
import io
import numpy as np, pandas as pd, soundfile as sf, librosa
from huggingface_hub import list_repo_files, hf_hub_download
from baseline_common import *

SEED, N_PER_DATASET = 42, 1200
os.makedirs(AUDIO_DIR, exist_ok=True)


def to_16k(audio, sr):
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    return audio if sr == SAMPLE_RATE else librosa.resample(audio, orig_sr=sr, target_sr=SAMPLE_RATE)


def save_audio(clip_id, audio):
    path = audio_path(clip_id)
    if not os.path.exists(path):
        sf.write(path, audio, SAMPLE_RATE, subtype="FLOAT")
    return round(len(audio) / SAMPLE_RATE, 2)


def hf_download(repo, path, **kw):
    for attempt in range(10):
        try:
            return hf_hub_download(repo, path, repo_type="dataset", **kw)
        except Exception as e:
            if "429" not in str(e):
                raise
            print(f"  rate limited, waiting {30 * (attempt + 1)} s", flush=True)
            time.sleep(30 * (attempt + 1))
    raise RuntimeError(f"still rate limited: {path}")


def fleurs_clips():
    files = list_repo_files("google/fleurs", repo_type="dataset", revision="refs/convert/parquet")
    parts = []
    for split in ["test", "validation", "train"]:
        paths = sorted(f for f in files if f.startswith(f"ur_pk/{split}/") and f.endswith(".parquet"))
        df = pd.concat([pd.read_parquet(hf_download("google/fleurs", p, revision="refs/convert/parquet"))
                        for p in paths], ignore_index=True)
        df["split"] = split
        parts.append(df)
        print(f"  FLEURS {split}: {len(df)} rows")
    pool = pd.concat(parts, ignore_index=True)
    order = np.random.default_rng(SEED).permutation(len(pool))[:N_PER_DATASET]
    rows = []
    for i in order:
        r = pool.iloc[i]
        name = r["path"] if "path" in pool.columns and r["path"] else r["audio"]["path"]
        clip_id = f"fleurs/{r['split']}/{os.path.basename(name).rsplit('.', 1)[0]}"
        audio, sr = sf.read(io.BytesIO(r["audio"]["bytes"]), dtype="float32")
        rows.append({"clip_id": clip_id, "dataset": "FLEURS", "subset": r["split"], "category": "",
                     "duration_s": save_audio(clip_id, to_16k(audio, sr)), "reference": r["transcription"]})
    return rows


def urduspeech_clips():
    repo = "ASLP-lab/UrduSpeech"
    files = list_repo_files(repo, repo_type="dataset")
    jsonls = [f for root in ["benchmark/US-benchmark-Std/", "corpus/US-Std/"] for f in files
              if f.startswith(root) and f.endswith(".jsonl") and "paralinguistic" not in f.lower()
              and any(w in f.lower() for w in ("news", "podcast"))]
    audio_by_dir = {}
    for f in files:
        if f.lower().endswith(".wav"):
            audio_by_dir.setdefault(f.rsplit("/audio/", 1)[0], {})[f.rsplit("/", 1)[-1].lower()] = f
    rows = []
    for jf in jsonls:
        if len(rows) >= N_PER_DATASET:
            break
        audio_index = audio_by_dir.get(jf.rsplit("/", 1)[0], {})
        if not audio_index:
            continue
        for line in open(hf_download(repo, jf), encoding="utf-8"):
            line = line.strip()
            if not line:
                continue
            try:
                e = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = e.get("ground_truth") or e.get("Transcription")
            name = (e.get("Audio_Clip") or e.get("audio_id") or "").rsplit("/", 1)[-1].lower()
            wav = audio_index.get(name)
            if not text or not str(text).strip() or wav is None:
                continue
            clip_id = "urduspeech/" + wav.rsplit(".", 1)[0]
            audio, sr = sf.read(hf_download(repo, wav), dtype="float32")
            rows.append({"clip_id": clip_id, "dataset": "UrduSpeech",
                         "subset": "benchmark" if jf.startswith("benchmark/") else "corpus",
                         "category": e.get("Audio_category", ""),
                         "duration_s": save_audio(clip_id, to_16k(audio, sr)), "reference": str(text)})
            if len(rows) >= N_PER_DATASET:
                break
    return rows


clips = pd.DataFrame(fleurs_clips() + urduspeech_clips())
assert clips.clip_id.is_unique, "clip_id is not unique"
clips["included"] = clips.duration_s <= MAX_CLIP_SECONDS
clips["excluded_reason"] = np.where(clips.included, "", f"longer than {MAX_CLIP_SECONDS} s")

os.makedirs(DATA, exist_ok=True)
for dataset, path in SAMPLE_FILES.items():
    part = clips[clips.dataset == dataset]
    earlier = glob.glob(f"/kaggle/input/**/data/{os.path.basename(path)}", recursive=True)
    if earlier:
        assert pd.read_csv(earlier[0]).clip_id.tolist() == part.clip_id.tolist(), \
            f"{dataset} clips differ from the earlier run of this notebook"
    part.to_csv(path, index=False, encoding="utf-8")

print("\nClips per dataset and subset:")
clips["over_30_s"] = clips.duration_s > 30
print(clips.groupby(["dataset", "subset", "category"]).agg(
    clips=("clip_id", "size"), included=("included", "sum"), over_30_s=("over_30_s", "sum"),
    median_s=("duration_s", "median"), longest_s=("duration_s", "max")).to_string())
clips = clips.drop(columns="over_30_s")

## 3. Omnilingual and Whisper (Python 3.10 environment)

| Recogniser | Decoding | Word confidence |
|---|---|---|
| Omnilingual | `ASRInferencePipeline` defaults, one clip at a time | Not available: the pipeline computes beam scores internally but returns only text |
| Whisper | Model's default generation settings (greedy), language Urdu | Saved for clips up to 30 s: each word's probability is the product of its tokens' probabilities |

Whisper clips longer than 30 s use Whisper's long-form mode, which transcribes 30 s windows in sequence.
A Whisper output that stops at the 448-token limit instead of an end token is saved with status `cut_off`.

In [ ]:
%%writefile transcribe_310.py
import sys
import numpy as np, soundfile as sf, torch
from baseline_common import SAMPLE_RATE, run_recogniser


def read(path):
    return sf.read(path, dtype="float32")[0]


def omnilingual():
    from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
    pipe = ASRInferencePipeline(model_card="omniASR_LLM_1B_v2", device="cuda", dtype=torch.float16)
    beam = {k: repr(v) for k, v in vars(pipe).items() if "beam" in k.lower()}
    print("Omnilingual beam settings:", beam or "none exposed")
    decoding = f"pipeline defaults; {beam}" if beam else "pipeline defaults"

    def one(path, duration_s):
        text = pipe.transcribe([{"waveform": read(path), "sample_rate": SAMPLE_RATE}],
                               lang=["urd_Arab"], batch_size=1)[0]
        return {"text": text, "decoding": decoding, "word_confidence": None, "status": "ok"}

    run_recogniser("omnilingual_llm_1b_v2", one)


def whisper():
    from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq
    name = "openai/whisper-large-v3"
    proc = AutoProcessor.from_pretrained(name)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(name, torch_dtype=torch.float16,
                                                      low_cpu_mem_usage=True).to("cuda").eval()
    tok = proc.tokenizer
    special = set(tok.all_special_ids)
    eos = model.generation_config.eos_token_id
    beams = model.generation_config.num_beams or 1
    print(f"Whisper generation: num_beams={beams}, max_length={model.generation_config.max_length}")

    def word_confidence(sequence, scores):
        # Returns [[word, probability], ...]. A new word starts at a token that begins with a space;
        # a word's probability is the product of its tokens' probabilities. Special tokens are skipped.
        generated = sequence[-len(scores):].tolist()
        words, ids, prob = [], [], 1.0
        for token, step in zip(generated, scores):
            if token in special:
                continue
            p = torch.softmax(step[0].float(), dim=-1)[token].item()
            if tok.convert_ids_to_tokens(token).startswith("Ġ") and ids:
                words.append([tok.decode(ids).strip(), round(prob, 4)])
                ids, prob = [], 1.0
            ids.append(token)
            prob *= p
        if ids:
            words.append([tok.decode(ids).strip(), round(prob, 4)])
        return words

    def one(path, duration_s):
        audio = read(path)
        if duration_s <= 30:
            feats = proc(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt").input_features.to("cuda", torch.float16)
            with torch.no_grad():
                out = model.generate(feats, language="urdu", task="transcribe",
                                     return_dict_in_generate=True, output_scores=True)
            seq = out.sequences[0]
            return {"text": tok.decode(seq, skip_special_tokens=True),
                    "decoding": f"greedy, num_beams={beams}",
                    "word_confidence": word_confidence(seq, out.scores),
                    "status": "ok" if seq[-1].item() == eos else "cut_off"}
        inputs = proc(audio, sampling_rate=SAMPLE_RATE, return_tensors="pt", truncation=False,
                      padding="longest", return_attention_mask=True).to("cuda", torch.float16)
        with torch.no_grad():
            ids = model.generate(**inputs, language="urdu", task="transcribe", return_timestamps=True)
        return {"text": proc.batch_decode(ids, skip_special_tokens=True)[0],
                "decoding": f"long-form 30 s windows, num_beams={beams}",
                "word_confidence": None, "status": "ok"}

    run_recogniser("whisper_large_v3", one)


if __name__ == "__main__":
    {"omnilingual": omnilingual, "whisper": whisper}[sys.argv[1]]()

In [ ]:
sh(f"cd /kaggle/working && {PY} -u transcribe_310.py omnilingual")

In [ ]:
sh(f"cd /kaggle/working && {PY} -u transcribe_310.py whisper")

## 4. Indic Transcribe Core (default Python)

Language `ur`. Every clip is transcribed in one pass with `asr(path, lang="ur")`.

In [ ]:
import torch
from huggingface_hub import snapshot_download

model_dir = snapshot_download("bodhan-ai/indic-transcribe-core", ignore_patterns=["nemo/*", "*.png"])
sys.path.insert(0, model_dir)
from indic_transcribe import IndicTranscribe

asr = IndicTranscribe.from_pretrained(model_dir)
if hasattr(asr, "to"):
    asr = asr.to("cuda")

In [ ]:
def as_text(output):
    if isinstance(output, str):
        return output
    if isinstance(output, (list, tuple)):
        return as_text(output[0])
    if isinstance(output, dict):
        return output.get("text") or output.get("transcript") or str(output)
    return getattr(output, "text", str(output))


def indic_one(path, duration_s):
    return {"text": as_text(asr(path, lang="ur")), "decoding": "single pass, model defaults",
            "word_confidence": None, "status": "ok"}


run_recogniser("indic_transcribe_core", indic_one)
del asr
torch.cuda.empty_cache()

## 5. Scores

All scores use `clean_urdu` v1 on both the reference and the transcript.

- **WER** = (substitutions + deletions + insertions) / reference words, summed over all clips in the subset (corpus WER).
- **CER** = the same on characters.
- **S / D / I %** = substitutions, deletions and insertions as a percentage of reference words, so S + D + I = WER.
- A failed or empty transcript is scored as empty: every reference word counts as a deletion.
- Clips whose reference is empty after normalisation cannot be scored and are left out (the count is printed).
- Each subset is scored twice: all clips, and only clips whose reference has 3 or more words.

In [ ]:
import jiwer

clips = load_samples(included_only=True)
clips["reference_clean"] = clips.reference.map(clean_urdu)
clips["reference_words"] = clips.reference_clean.str.split().str.len()

by_clip = clips[["clip_id", "dataset", "subset", "category", "duration_s", "reference", "reference_clean"]].copy()
COUNTS = {}
for stem, name in RECOGNISERS.items():
    t = load_transcripts(stem).set_index("clip_id")
    missing = set(clips.clip_id) - set(t.index)
    assert not missing, f"{name} has no transcript for {len(missing)} clips; run its step again"
    t = t.loc[clips.clip_id]
    by_clip[stem] = t.transcript.values
    by_clip[f"{stem}_clean"] = [clean_urdu(x) for x in t.transcript]
    by_clip[f"{stem}_time_s"] = t.time_s.astype(float).values
    by_clip[f"{stem}_status"] = t.status.values


def edit_counts(ref, hyp):
    # Returns (S, D, I, reference words, character edits, reference characters).
    if not hyp:
        return 0, len(ref.split()), 0, len(ref.split()), len(ref), len(ref)
    w = jiwer.process_words(ref, hyp)
    c = jiwer.process_characters(ref, hyp)
    return (w.substitutions, w.deletions, w.insertions, w.hits + w.substitutions + w.deletions,
            c.substitutions + c.deletions + c.insertions, c.hits + c.substitutions + c.deletions)


scorable = by_clip.reference_clean.str.len() > 0
print(f"Clips with an empty reference after normalisation (not scored): {(~scorable).sum()}")
for stem in RECOGNISERS:
    counts = np.array([edit_counts(r, h) if ok else (0, 0, 0, 0, 0, 0)
                       for r, h, ok in zip(by_clip.reference_clean, by_clip[f"{stem}_clean"], scorable)])
    COUNTS[stem] = counts
    with np.errstate(invalid="ignore", divide="ignore"):
        by_clip[f"{stem}_WER_%"] = np.round(100 * counts[:, :3].sum(1) / counts[:, 3], 2)
        by_clip[f"{stem}_CER_%"] = np.round(100 * counts[:, 4] / counts[:, 5], 2)

by_clip.to_csv(f"{DATA}/transcripts/all_recognisers_by_clip.csv", index=False, encoding="utf-8-sig")

In [ ]:
is_fleurs, is_us = by_clip.dataset.eq("FLEURS"), by_clip.dataset.eq("UrduSpeech")
SUBSETS = [
    ("FLEURS", "test + validation", is_fleurs & by_clip.subset.isin(["test", "validation"])),
    ("FLEURS", "train", is_fleurs & by_clip.subset.eq("train")),
    ("FLEURS", "test + validation + train", is_fleurs),
    ("UrduSpeech", "benchmark (human-checked references)", is_us & by_clip.subset.eq("benchmark")),
    ("UrduSpeech", "corpus (Gemini references)", is_us & by_clip.subset.eq("corpus")),
    ("UrduSpeech", "benchmark + corpus", is_us),
]
FILTERS = [("all clips", scorable), ("references with 3+ words", scorable & (clips.reference_words >= 3))]

rows = []
for dataset, subset, in_subset in SUBSETS:
    for filter_name, keep in FILTERS:
        m = (in_subset & keep).values
        for stem, name in RECOGNISERS.items():
            c = COUNTS[stem][m].sum(0)
            times = by_clip.loc[m, f"{stem}_time_s"]
            rows.append({"dataset": dataset, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                         "reference_words": int(c[3]), "recogniser": name,
                         "WER_%": round(100 * c[:3].sum() / c[3], 2), "CER_%": round(100 * c[4] / c[5], 2),
                         "substitutions_%": round(100 * c[0] / c[3], 2), "deletions_%": round(100 * c[1] / c[3], 2),
                         "insertions_%": round(100 * c[2] / c[3], 2),
                         "failed_or_empty": int((by_clip.loc[m, f"{stem}_status"] != "ok").sum()),
                         "total_time_min": round(times.sum() / 60, 1), "seconds_per_clip": round(times.mean(), 2)})
scores = pd.DataFrame(rows)
os.makedirs(f"{RESULTS}/scores", exist_ok=True)
scores.to_csv(f"{RESULTS}/scores/recogniser_scores.csv", index=False, encoding="utf-8-sig")

show = ["recogniser", "clips", "WER_%", "CER_%", "substitutions_%", "deletions_%", "insertions_%",
        "failed_or_empty", "seconds_per_clip", "total_time_min"]
for (dataset, subset, filter_name), table in scores.groupby(["dataset", "subset", "clips_scored"], sort=False):
    print(f"\n{dataset} | {subset} | {filter_name}")
    print(table[show].to_string(index=False))

## 6. Paired bootstrap

For each pair of recognisers, on the same clips:
1. Draw as many clips as the subset has, at random with replacement.
2. Compute WER(A) − WER(B) on that sample.
3. Repeat 1,000 times (seed 42). The middle 95% of the differences is the 95% interval.

If the whole interval is below 0, A has the lower WER; if above 0, B does; if it contains 0, there is no reliable difference.

In [ ]:
from itertools import combinations

def paired_bootstrap(errors_a, errors_b, words, n=1000, seed=42):
    idx = np.random.default_rng(seed).integers(0, len(words), size=(n, len(words)))
    diff = 100 * (errors_a[idx].sum(1) - errors_b[idx].sum(1)) / words[idx].sum(1)
    return np.percentile(diff, [2.5, 97.5])

rows = []
for dataset, subset, in_subset in SUBSETS:
    for filter_name, keep in FILTERS:
        m = (in_subset & keep).values
        for a, b in combinations(RECOGNISERS, 2):
            ea, eb = COUNTS[a][m, :3].sum(1), COUNTS[b][m, :3].sum(1)
            words = COUNTS[a][m, 3]
            lo, hi = paired_bootstrap(ea, eb, words)
            wa, wb = 100 * ea.sum() / words.sum(), 100 * eb.sum() / words.sum()
            verdict = (f"{RECOGNISERS[a]} lower" if hi < 0 else f"{RECOGNISERS[b]} lower" if lo > 0
                       else "no reliable difference")
            rows.append({"dataset": dataset, "subset": subset, "clips_scored": filter_name, "clips": int(m.sum()),
                         "recogniser_A": RECOGNISERS[a], "recogniser_B": RECOGNISERS[b],
                         "WER_A_%": round(wa, 2), "WER_B_%": round(wb, 2), "difference_A_minus_B": round(wa - wb, 2),
                         "interval_low": round(lo, 2), "interval_high": round(hi, 2), "result": verdict})
boot = pd.DataFrame(rows)
boot.to_csv(f"{RESULTS}/scores/paired_bootstrap.csv", index=False, encoding="utf-8-sig")

show = ["recogniser_A", "recogniser_B", "WER_A_%", "WER_B_%", "difference_A_minus_B", "interval_low", "interval_high", "result"]
for (dataset, subset, filter_name), table in boot.groupby(["dataset", "subset", "clips_scored"], sort=False):
    if filter_name == "all clips":
        print(f"\n{dataset} | {subset} | {filter_name}")
        print(table[show].to_string(index=False))
print("\nThe same comparisons for references with 3+ words are in results/scores/paired_bootstrap.csv")

## 7. Three example clips (UrduSpeech benchmark)

In [ ]:
from IPython.display import HTML, display

style = "font-family:'Noto Nastaliq Urdu','Noto Naskh Arabic',serif;font-size:17px;direction:rtl;text-align:right;padding:2px 8px"
examples = by_clip[by_clip.subset.eq("benchmark")].head(3)
for _, r in examples.iterrows():
    html = f"<b>{r.clip_id}</b> ({r.duration_s} s)"
    for label, col in [("Reference", "reference")] + [(n, s) for s, n in RECOGNISERS.items()]:
        wer = "" if col == "reference" else f" — WER {r[f'{col}_WER_%']}%"
        html += f"<div style='font-size:11px;color:#777'>{label}{wer}</div><div style='{style}'>{r[col]}</div>"
    display(HTML(html + "<hr>"))

## 8. Run information

In [ ]:
import transformers

env310 = subprocess.run(f"{PY} -c \"import torch, transformers, omnilingual_asr as o; "
                        f"print(torch.__version__, transformers.__version__, getattr(o, '__version__', '0.2.0'))\"",
                        shell=True, capture_output=True, text=True).stdout.split()
gpu = subprocess.run("nvidia-smi --query-gpu=name,driver_version --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
all_clips = load_samples()
info = {
    "machine": f"Kaggle, {gpu.splitlines()[0]}",
    "normaliser": NORMALISER_VERSION,
    "max_clip_seconds": MAX_CLIP_SECONDS,
    "clips_total": len(all_clips),
    "clips_transcribed": int((all_clips.included.astype(str) == "True").sum()),
    "clips_excluded": all_clips.loc[all_clips.included.astype(str) != "True", "clip_id"].tolist(),
    "environments": {
        "python_3_10 (Omnilingual, Whisper)": {"torch": env310[0], "transformers": env310[1], "omnilingual_asr": env310[2]},
        "kaggle_default (Indic Transcribe)": {"python": sys.version.split()[0], "torch": torch.__version__,
                                              "transformers": transformers.__version__},
    },
    "models": {"whisper_large_v3": "openai/whisper-large-v3", "omnilingual_llm_1b_v2": "omniASR_LLM_1B_v2",
               "indic_transcribe_core": "bodhan-ai/indic-transcribe-core"},
    "decoding": {stem: sorted(load_transcripts(stem).decoding.unique().tolist()) for stem in RECOGNISERS},
    "elapsed_minutes": {stem: round(load_transcripts(stem).time_s.astype(float).sum() / 60, 1) for stem in RECOGNISERS},
    "bootstrap": {"resamples": 1000, "seed": 42},
}
json.dump(info, open(f"{RESULTS}/run_info.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
print(json.dumps(info, indent=2, ensure_ascii=False))

shutil.make_archive("/kaggle/working/output", "zip", OUT)
print("\nSaved /kaggle/working/output.zip")